In [ ]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

[Introduction](introyt1_tutorial.html) \|\|
[Tensors](tensors_deeper_tutorial.html) \|\|
[Autograd](autogradyt_tutorial.html) \|\| [Building
Models](modelsyt_tutorial.html) \|\| [TensorBoard
Support](tensorboardyt_tutorial.html) \|\| [Training
Models](trainingyt.html) \|\| **Model Understanding**

Model Understanding with Captum  
使用 Captum 理解模型
===============================

Follow along with the video below or on
[youtube](https://www.youtube.com/watch?v=Am2EF9CLu-g). Download the
notebook and corresponding files
[here](https://pytorch-tutorial-assets.s3.amazonaws.com/youtube-series/video7.zip).  
请跟随下方的视频进行操作，或者在 [YouTube]() 上观看。
你可以点击 [此处]() 下载配套的 Notebook 笔记本及相应的文件。

In [ ]:
# Run this cell to load the video
from IPython.display import display, HTML
html_code = """
<div style="margin-top:10px; margin-bottom:10px;">
  <iframe width="560" height="315" src="https://www.youtube.com/embed/Am2EF9CLu-g" frameborder="0" allow="accelerometer; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>
</div>
"""
display(HTML(html_code))



[Captum](https://captum.ai/) ("comprehension" in Latin) is an open
source, extensible library for model interpretability built on PyTorch.  
[Captum]()（在拉丁语中意为“理解”）是一个建立在 PyTorch 之上的开源、可扩展的模型可解释性库。

With the increase in model complexity and the resulting lack of
transparency, model interpretability methods have become increasingly
important. Model understanding is both an active area of research as
well as an area of focus for practical applications across industries
using machine learning. Captum provides state-of-the-art algorithms,
including Integrated Gradients, to provide researchers and developers
with an easy way to understand which features are contributing to a
model's output.  
随着模型复杂性的增加以及由此带来的不透明性，模型可解释性方法变得越来越重要。模型理解不仅是当前活跃的研究领域，也是各行业在机器学习实际应用中关注的重点。Captum 提供了包括集成梯度（Integrated Gradients）在内的前沿算法，为研究人员和开发者提供了一种简便的方法，以了解哪些特征对模型的输出做出了贡献。

Full documentation, an API reference, and a suite of tutorials on
specific topics are available at the [captum.ai](https://captum.ai/)
website.  
完整的文档、API 参考以及一系列针对特定主题的教程，均可在 [captum.ai](https://captum.ai/) 网站上获取。







Introduction  
简介
------------

Captum's approach to model interpretability is in terms of
*attributions.* There are three kinds of attributions available in
Captum:  
Captum 对模型可解释性的方法是以**归因（Attributions）**为基础的。Captum 提供了三种类型的归因：

-   **Feature Attribution** seeks to explain a particular output in
    terms of features of the input that generated it. Explaining whether
    a movie review was positive or negative in terms of certain words in
    the review is an example of feature attribution.  
    **特征归因（Feature Attribution）**：试图解释特定输出是由输入的哪些特征生成的。例如，通过评论中的某些词语来解释一条电影评论是积极的还是消极的，这就是特征归因的一个例子。
-   **Layer Attribution** examines the activity of a model's hidden
    layer subsequent to a particular input. Examining the
    spatially-mapped output of a convolutional layer in response to an
    input image in an example of layer attribution.  
    **层归因（Layer Attribution）**：检查模型隐藏层在特定输入后的活动。例如，检查卷积层对输入图像的响应及其空间映射输出，这就是层归因的一个例子。
-   **Neuron Attribution** is analagous to layer attribution, but
    focuses on the activity of a single neuron.  
    **神经元归因（Neuron Attribution）**：类似于层归因，但专注于模型中单个神经元的活动。

In this interactive notebook, we'll look at Feature Attribution and
Layer Attribution.  
在这个交互式笔记本中，我们将重点探讨特征归因和层归因。

Each of the three attribution types has multiple **attribution
algorithms** associated with it. Many attribution algorithms fall into
two broad categories:  
这三种归因类型中的每一种都关联着多个**归因算法**。许多归因算法可以分为两大类：

-   **Gradient-based algorithms** calculate the backward gradients of a
    model output, layer output, or neuron activation with respect to the
    input. **Integrated Gradients** (for features), **Layer Gradient \*
    Activation**, and **Neuron Conductance** are all gradient-based
    algorithms.  
    **基于梯度的算法（Gradient-based algorithms）**：计算模型输出、层输出或神经元激活相对于输入的反向梯度。**集成梯度（Integrated Gradients）**（用于特征）、**层梯度 * 激活（Layer Gradient * Activation）**和**神经元电导（Neuron Conductance）**都是基于梯度的算法。
-   **Perturbation-based algorithms** examine the changes in the output
    of a model, layer, or neuron in response to changes in the input.
    The input perturbations may be directed or random. **Occlusion,**
    **Feature Ablation,** and **Feature Permutation** are all
    perturbation-based algorithms.  
    **基于扰动的算法（Perturbation-based algorithms）**：检查模型、层或神经元对输入变化的响应。输入扰动可能是有方向的，也可能是随机的。**遮挡（Occlusion）**、**特征消融（Feature Ablation）**和**特征置换（Feature Permutation）**都属于基于扰动的算法。


We'll be examining algorithms of both types below.  
接下来，我们将分别探讨这两种类型的算法。

Especially where large models are involved, it can be valuable to
visualize attribution data in ways that relate it easily to the input
features being examined. While it is certainly possible to create your
own visualizations with Matplotlib, Plotly, or similar tools, Captum
offers enhanced tools specific to its attributions:  
特别是对于大型模型，将归因数据以易于与被检查的输入特征相关联的方式进行可视化，是非常有价值的。虽然你当然可以使用 Matplotlib、Plotly 或类似工具创建自己的可视化图表，但 Captum 提供了专门针对其归因结果的增强工具：

-   The `captum.attr.visualization` module (imported below as `viz`)
    provides helpful functions for visualizing attributions related to
    images.  
    `captum.attr.visualization` 模块（在下方导入为 `viz`）提供了用于可视化与图像相关的归因结果的实用函数。

This visualization toolset will be demonstrated throughout this
notebook.  
本笔记本将全程演示这套可视化工具。

Installation  
安装
------------

Before you get started, you need to have a Python environment with:  
在开始之前，你需要准备一个包含以下内容的 Python 环境：

-   Python version 3.9 or higher  
    Python 3.9 或更高版本
-   PyTorch (the latest version is recommended)  
    PyTorch（推荐最新版本）
-   TorchVision (the latest version is recommended)  
    TorchVision（推荐最新版本）
-   Captum (the latest version is recommended)  
    Captum（推荐最新版本）
-   Matplotlib (the latest version is recommended)  
    Matplotlib（推荐最新版本）

To install Captum in a virtual environment, use:
要在虚拟环境中安装 Captum，请使用：

``` {.sh}
pip install torch torchvision captum matplotlib
```

Restart this notebook in the environment you set up, and you're ready to
go!  
在你设置好的环境中重新启动此笔记本，就可以开始了！

A First Example  
第一个示例
---------------

To start, let's take a simple, visual example. We'll start with a ResNet
model pretrained on the ImageNet dataset. We'll get a test input, and
use different **Feature Attribution** algorithms to examine how the
input images affect the output, and see a helpful visualization of this
input attribution map for some test images.  
首先，让我们从一个简单的视觉示例开始。我们将使用在 ImageNet 数据集上预训练的 ResNet 模型。我们将获取一个测试输入，并使用不同的**特征归因（Feature Attribution）**算法来检查输入图像对输出的影响，同时查看一些测试图像的输入归因映射的有用可视化结果。

First, some imports:  
首先，导入一些必要的库：
``` {.sh}
pip install torch torchvision captum matplotlib
```

In [ ]:
import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.models as models

import captum
from captum.attr import IntegratedGradients, Occlusion, LayerGradCam, LayerAttribution
from captum.attr import visualization as viz

import os, sys
import json

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

Now we'll use the TorchVision model library to download a pretrained
ResNet. Since we're not training, we'll place it in evaluation mode for
now.  
现在，我们将使用 TorchVision 模型库下载一个预训练的 ResNet 模型。由于我们目前不进行训练，所以先将其置于评估模式。

In [ ]:
model = models.resnet18(weights='IMAGENET1K_V1')
model = model.eval()

The place where you got this interactive notebook should also have an
`img` folder with a file `cat.jpg` in it.  
获取此交互式笔记本的地方应该还有一个 `img` 文件夹，其中包含一个名为 `cat.jpg` 的文件。

In [ ]:
test_img = Image.open('img/cat.jpg')
test_img_data = np.asarray(test_img)
plt.imshow(test_img_data)
plt.show()

Our ResNet model was trained on the ImageNet dataset, and expects images
to be of a certain size, with the channel data normalized to a specific
range of values. We'll also pull in the list of human-readable labels
for the categories our model recognizes - that should be in the `img`
folder as well.  
我们的 ResNet 模型是在 ImageNet 数据集上训练的，它要求图像具有特定的尺寸，并且通道数据需归一化到特定的值范围。我们还会提取模型所识别类别的可读标签列表，该列表同样位于 `img` 文件夹中。

In [ ]:
# model expects 224x224 3-color image
transform = transforms.Compose([
 transforms.Resize(224),
 transforms.CenterCrop(224),
 transforms.ToTensor()
])

# standard ImageNet normalization
transform_normalize = transforms.Normalize(
     mean=[0.485, 0.456, 0.406],
     std=[0.229, 0.224, 0.225]
 )

transformed_img = transform(test_img)
input_img = transform_normalize(transformed_img)
input_img = input_img.unsqueeze(0) # the model requires a dummy batch dimension

labels_path = 'img/imagenet_class_index.json'
with open(labels_path) as json_data:
    idx_to_labels = json.load(json_data)

Now, we can ask the question: What does our model think this image
represents?  
现在，我们可以提出一个问题：我们的模型认为这张图像代表什么？

In [ ]:
output = model(input_img)
output = F.softmax(output, dim=1)
prediction_score, pred_label_idx = torch.topk(output, 1)
pred_label_idx.squeeze_()
predicted_label = idx_to_labels[str(pred_label_idx.item())][1]
print('Predicted:', predicted_label, '(', prediction_score.squeeze().item(), ')')

We've confirmed that ResNet thinks our image of a cat is, in fact, a
cat. But *why* does the model think this is an image of a cat?  
我们已经确认，ResNet 认为我们这张猫的图片确实是一只猫。但是，模型*为什么*会认为这是一张猫的图片呢？

For the answer to that, we turn to Captum.  
为了解答这个问题，我们转向 Captum。

Feature Attribution with Integrated Gradients  
使用集成梯度进行特征归因
=============================================

**Feature attribution** attributes a particular output to features of
the input. It uses a specific input - here, our test image - to generate
a map of the relative importance of each input feature to a particular
output feature.  
**特征归因**将特定的输出归因于输入的各个特征。它使用特定的输入（在这里是我们的测试图像）来生成一张映射图，展示每个输入特征对特定输出特征的相对重要性。


[Integrated Gradients](https://captum.ai/api/integrated_gradients.html)
is one of the feature attribution algorithms available in Captum.
Integrated Gradients assigns an importance score to each input feature
by approximating the integral of the gradients of the model's output
with respect to the inputs.  
[集成梯度（Integrated Gradients）](https://captum.ai/api/integrated_gradients.html)是 Captum 中可用的一种特征归因算法。集成梯度通过近似计算模型输出相对于输入的梯度积分，为每个输入特征分配一个重要性得分。

In our case, we're going to be taking a specific element of the output
vector - that is, the one indicating the model's confidence in its
chosen category - and use Integrated Gradients to understand what parts
of the input image contributed to this output.  
在我们的案例中，我们将提取输出向量中的一个特定元素——即表明模型对其所选类别置信度的那个元素——并使用集成梯度来理解输入图像的哪些部分对这一输出做出了贡献。

Once we have the importance map from Integrated Gradients, we'll use the
visualization tools in Captum to give a helpful representation of the
importance map. Captum's `visualize_image_attr()` function provides a
variety of options for customizing display of your attribution data.
Here, we pass in a custom Matplotlib color map.  
一旦我们通过集成梯度获得了重要性映射图，我们将使用 Captum 中的可视化工具来直观地呈现该映射图。Captum 的 `visualize_image_attr()` 函数提供了多种选项，用于自定义归因数据的显示。在这里，我们传入了一个自定义的 Matplotlib 颜色映射（color map）。

Running the cell with the `integrated_gradients.attribute()` call will
usually take a minute or two.  
运行包含 `integrated_gradients.attribute()` 调用的单元格通常需要一到两分钟的时间。

In [ ]:
# Initialize the attribution algorithm with the model
integrated_gradients = IntegratedGradients(model)

# Ask the algorithm to attribute our output target to 
attributions_ig = integrated_gradients.attribute(input_img, target=pred_label_idx, n_steps=200)

# Show the original image for comparison
_ = viz.visualize_image_attr(None, np.transpose(transformed_img.squeeze().cpu().detach().numpy(), (1,2,0)),
                      method="original_image", title="Original Image")

default_cmap = LinearSegmentedColormap.from_list('custom blue', 
                                                 [(0, '#ffffff'),
                                                  (0.25, '#0000ff'),
                                                  (1, '#0000ff')], N=256)

_ = viz.visualize_image_attr(np.transpose(attributions_ig.squeeze().cpu().detach().numpy(), (1,2,0)),
                             np.transpose(transformed_img.squeeze().cpu().detach().numpy(), (1,2,0)),
                             method='heat_map',
                             cmap=default_cmap,
                             show_colorbar=True,
                             sign='positive',
                             title='Integrated Gradients')

In the image above, you should see that Integrated Gradients gives us
the strongest signal around the cat's location in the image.  
在上图中，你可以看到集成梯度（Integrated Gradients）在图像中猫的位置周围给出了最强的信号。

Feature Attribution with Occlusion  
使用遮挡（Occlusion）进行特征归因
==================================

Gradient-based attribution methods help to understand the model in terms
of directly computing out the output changes with respect to the input.
*Perturbation-based attribution* methods approach this more directly, by
introducing changes to the input to measure the effect on the output.
[Occlusion](https://captum.ai/api/occlusion.html) is one such method. It
involves replacing sections of the input image, and examining the effect
on the output signal.

Below, we set up Occlusion attribution. Similarly to configuring a
convolutional neural network, you can specify the size of the target
region, and a stride length to determine the spacing of individual
measurements. We'll visualize the output of our Occlusion attribution
with `visualize_image_attr_multiple()`, showing heat maps of both
positive and negative attribution by region, and by masking the original
image with the positive attribution regions. The masking gives a very
instructive view of what regions of our cat photo the model found to be
most "cat-like".

==================================

基于梯度的归因方法通过直接计算输出相对于输入的变化来帮助理解模型。而**基于扰动的归因**方法则通过直接改变输入来衡量其对输出的影响，从而更加直观地实现这一目的。<遮挡（Occlusion）>就是其中一种方法。它涉及替换输入图像的局部区域，并检查这对输出信号产生的影响。

在下方，我们配置了遮挡归因。与配置卷积神经网络类似，你可以指定目标区域的大小以及步幅（stride length），以确定各个测量点之间的间距。我们将使用 `visualize_image_attr_multiple()` 函数来可视化遮挡归因的输出，同时展示各个区域正向和负向归因的热力图，并通过正向归因区域对原始图像进行掩码处理。这种掩码处理方式能够非常直观地展示模型认为我们这张猫照片中哪些区域最具有“猫”的特征。


In [ ]:
occlusion = Occlusion(model)

attributions_occ = occlusion.attribute(input_img,
                                       target=pred_label_idx,
                                       strides=(3, 8, 8),
                                       sliding_window_shapes=(3,15, 15),
                                       baselines=0)


_ = viz.visualize_image_attr_multiple(np.transpose(attributions_occ.squeeze().cpu().detach().numpy(), (1,2,0)),
                                      np.transpose(transformed_img.squeeze().cpu().detach().numpy(), (1,2,0)),
                                      ["original_image", "heat_map", "heat_map", "masked_image"],
                                      ["all", "positive", "negative", "positive"],
                                      show_colorbar=True,
                                      titles=["Original", "Positive Attribution", "Negative Attribution", "Masked"],
                                      fig_size=(18, 6)
                                     )

Again, we see greater significance placed on the region of the image
that contains the cat.


Layer Attribution with Layer GradCAM  
使用 Layer GradCAM 进行层归因
====================================

**Layer Attribution** allows you to attribute the activity of hidden
layers within your model to features of your input. Below, we'll use a
layer attribution algorithm to examine the activity of one of the
convolutional layers within our model.  
**层归因（Layer Attribution）**允许你将模型内部隐藏层的活动归因于输入的特征。在下方，我们将使用一种层归因算法来检查模型中某个卷积层的活动。

GradCAM computes the gradients of the target output with respect to the
given layer, averages for each output channel (dimension 2 of output),
and multiplies the average gradient for each channel by the layer
activations. The results are summed over all channels. GradCAM is
designed for convnets; since the activity of convolutional layers often
maps spatially to the input, GradCAM attributions are often upsampled
and used to mask the input.  
GradCAM 会计算目标输出相对于给定层的梯度，对每个输出通道（即输出的第 2 维）进行平均，然后将每个通道的平均梯度乘以该层的激活值。最终的结果会在所有通道上求和。
GradCAM 是专为卷积神经网络设计的；由于卷积层的活动通常在空间上映射到输入，GradCAM 的归因结果通常会被上采样，并用于对输入进行掩码处理。

Layer attribution is set up similarly to input attribution, except that
in addition to the model, you must specify a hidden layer within the
model that you wish to examine. As above, when we call `attribute()`, we
specify the target class of interest.  
层归因的设置方式与输入归因类似，但除了指定模型之外，你还必须指定模型内你希望检查的隐藏层。与上文相同，当我们调用 `attribute()` 时，需要指定感兴趣的目标类别。

In [ ]:
layer_gradcam = LayerGradCam(model, model.layer3[1].conv2)
attributions_lgc = layer_gradcam.attribute(input_img, target=pred_label_idx)

_ = viz.visualize_image_attr(attributions_lgc[0].cpu().permute(1,2,0).detach().numpy(),
                             sign="all",
                             title="Layer 3 Block 1 Conv 2")

We'll use the convenience method `interpolate()` in the
[LayerAttribution](https://captum.ai/api/base_classes.html?highlight=layerattribution#captum.attr.LayerAttribution)
base class to upsample this attribution data for comparison to the input
image.  
我们将使用  基类中的便捷方法 `interpolate()` 对这些归因数据进行上采样，以便与输入图像进行对比。

In [ ]:
upsamp_attr_lgc = LayerAttribution.interpolate(attributions_lgc, input_img.shape[2:])

print(attributions_lgc.shape)
print(upsamp_attr_lgc.shape)
print(input_img.shape)

_ = viz.visualize_image_attr_multiple(upsamp_attr_lgc[0].cpu().permute(1,2,0).detach().numpy(),
                                      transformed_img.permute(1,2,0).numpy(),
                                      ["original_image","blended_heat_map","masked_image"],
                                      ["all","positive","positive"],
                                      show_colorbar=True,
                                      titles=["Original", "Positive Attribution", "Masked"],
                                      fig_size=(18, 6))

Visualizations such as this can give you novel insights into how your
hidden layers respond to your input.  
此类可视化能够为你提供全新的视角，帮助你深入了解隐藏层是如何对输入做出响应的。